# Investment AI: analyst conviction + pullback opportunities

One ranking follows your priorities: strong analyst agreement, target upside, then a moderate recent decline with initial stabilization. **A high target does not predict recovery speed.**

Run all cells for offline analysis. Change the settings below to experiment. Enable refresh only when you want new Yahoo requests. See `README.md` for the formula, thresholds and limitations.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from investment_scanner import Settings, DISPLAY_COLUMNS, refresh_cache, scan_cache, save_results

DATA_DIR = Path("sp500_notebook_data")
OUTPUT_DIR = DATA_DIR / "output"
REFRESH_DATA = False
REFRESH_SYMBOLS = None  # Example: ["AAPL", "MSFT"]; None updates the full universe.
settings = Settings(
    min_ratings=10,
    min_buy_pct=70,
    max_sell_pct=10,
    min_upside_pct=15,
    min_week_drop_pct=3,
    max_week_drop_pct=10,
    min_fortnight_drop_pct=5,
    max_fortnight_drop_pct=15,
)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)


In [ ]:
if REFRESH_DATA:
    refresh_errors = refresh_cache(DATA_DIR, settings, REFRESH_SYMBOLS)
    print(f"Refresh completed with {len(refresh_errors)} errors; see cache/refresh_errors.csv.")
else:
    print("Offline mode: using existing cached data. Dates are checked before shortlisting.")


In [ ]:
ranked = scan_cache(DATA_DIR, settings)
watchlist = ranked.loc[ranked.thesis_eligible].copy()
pullback_candidates = ranked.loc[ranked.status.eq("PULLBACK_CANDIDATE")].copy()
review_or_wait = watchlist.loc[~watchlist.status.eq("PULLBACK_CANDIDATE")].copy()
print(f"Scanned: {len(ranked)} | Analyst watchlist: {len(watchlist)} | Pullback candidates: {len(pullback_candidates)}")
display(ranked.status.value_counts().rename_axis("Status").to_frame("Stocks"))
if ranked.status.eq("DATA_REFRESH_REQUIRED").any():
    print("Stale or missing data excluded from current shortlists. Enable REFRESH_DATA to update it.")


## Analyst watchlist
Companies passing analyst support, upside, liquidity and freshness gates. Conviction bands come first. Inspect the status before treating a company as a pullback candidate.


In [ ]:
def show(frame, limit=30):
    columns = [column for column in DISPLAY_COLUMNS if column in frame]
    display(frame[columns].head(limit).round(2))

show(watchlist)


## Pullback candidates
These also pass the pullback, stabilization and risk checks. An empty shortlist means no cached stock currently meets every rule; thresholds are never loosened automatically.


In [ ]:
show(pullback_candidates)


## Wait or review
The reason column explains missing timing confirmation or risk checks. Unknown earnings/revision data is visible rather than treated as favorable.


In [ ]:
show(review_or_wait)


## Inspect one company
Use this for excluded stocks too. Analyst retrieval time is not the publication time of the underlying research. Target prices have no verified recovery deadline.


In [ ]:
SYMBOL = "AAPL"
inspect_columns = [c for c in DISPLAY_COLUMNS + [
    "target_source", "target_low", "target_high", "target_spread_pct",
    "target_low_upside_pct", "drawdown_20d_pct", "pullback_sigma", "stabilizing",
    "distance_to_200d_ma_pct", "eps_estimate_change_30d_pct", "days_to_earnings",
    "price_as_of", "fetched_at_utc", "price_age_days", "analyst_retrieval_age_days",
    "risk_flags",
] if c in ranked]
display(ranked.loc[ranked.symbol.eq(SYMBOL), inspect_columns].T)


## Save the same results
Exports include the full ranking, analyst watchlist, pullback shortlist, review/wait list, and a manifest containing the settings and scan date. No alternative ranking is applied during export.


In [ ]:
paths = save_results(ranked, OUTPUT_DIR, settings)
for name, path in paths.items():
    print(f"{name}: {path}")
